# Partie V — Modèles de diffusion

Un modèle de diffusion apprend à créer des images en **retirant progressivement du bruit**.

- **Diffusion directe** (sans réseau) : on ajoute du bruit gaussien à une image réelle, étape par étape,
  jusqu'à obtenir du bruit presque pur : $x_0 \rightarrow x_1 \rightarrow \dots \rightarrow x_T$.
- **Diffusion inverse** (apprise) : un réseau apprend à prédire le bruit présent dans une image au temps $t$.
  En partant de bruit pur et en le retirant pas à pas, on génère une nouvelle image.

## 1. Préparer les données et le périphérique

Comme pour le GAN, les pixels sont ramenés dans [−1, 1].

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Périphérique :", device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

## 2. Définir comment ajouter du bruit

`T = 200` niveaux de bruit, avec un calendrier cosinus :

| Coefficient | Signification |
|---|---|
| `betas[t]` | variance du bruit ajouté à l'étape t |
| `alphas[t] = 1 - betas[t]` | part du signal conservée à cette étape |
| `alpha_bars[t]` | produit cumulé des `alphas` |

`add_noise` bruite directement une image au temps $t$, sans passer par les étapes précédentes :
$x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon$.
Elle renvoie aussi le bruit $\varepsilon$ utilisé, qui sert de cible pendant l'entraînement.

In [ ]:
T = 200

def cosine_beta_schedule(timesteps, s=0.008):
    steps = torch.linspace(0, timesteps, timesteps + 1)

    alpha_bar = torch.cos(((steps / timesteps + s) / (1 + s)) * math.pi / 2).pow(2)
    alpha_bar = alpha_bar / alpha_bar[0]

    betas = 1 - alpha_bar[1:] / alpha_bar[:-1]

    return betas.clamp(min=1e-4, max=0.999)


betas = cosine_beta_schedule(T).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

alpha_bars_prev = torch.cat([torch.ones(1, device=device), alpha_bars[:-1]])

# Variance utilisée pendant la génération.
posterior_variance = betas * (1 - alpha_bars_prev) / (1 - alpha_bars)


def extract(values, t):
    """Sélectionner un coefficient par image et permettre le broadcasting."""
    return values[t].view(-1, 1, 1, 1)


def add_noise(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    signal = extract(alpha_bars.sqrt(), t)
    noise_scale = extract((1 - alpha_bars).sqrt(), t)

    xt = signal * x0 + noise_scale * noise

    return xt, noise

## 3. Observer la dégradation progressive d'une image

In [ ]:
x0, _ = train_dataset[0]
x0 = x0.unsqueeze(0).to(device)

times = [0, 40, 80, 120, 160, 199]
noise = torch.randn_like(x0)

fig, axes = plt.subplots(1, len(times) + 1, figsize=(14, 2))

axes[0].imshow(x0[0, 0].cpu(), cmap="gray", vmin=-1, vmax=1)
axes[0].set_title("Original")
axes[0].axis("off")

for ax, step in zip(axes[1:], times):
    t = torch.tensor([step], device=device)
    xt, _ = add_noise(x0, t, noise)

    ax.imshow(xt[0, 0].cpu(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t = {step}")
    ax.axis("off")

plt.tight_layout()
plt.show()

Au début, le chiffre est bien visible ; aux temps intermédiaires les détails disparaissent ; à la fin il ne reste
que du bruit. C'est ce processus que le réseau devra apprendre à inverser.

## 4. Construire le réseau qui prédit le bruit

Le modèle reçoit une image bruitée et le temps $t$, et prédit le bruit ajouté (même forme que l'image).

- `TimeEmbedding` : transforme $t$ en un vecteur de sinus et cosinus de différentes fréquences.
- `ConvBlock` : convolution, ajout d'une projection du temps (une valeur par canal), seconde convolution.
- `SmallUNet` : descend en résolution (28 → 14 → 7) puis remonte (7 → 14 → 28), avec des connexions qui
  recopient les cartes de la descente. Pas de Sigmoid en sortie : le bruit prédit peut être négatif.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2

        frequencies = torch.exp(
            -math.log(10000)
            * torch.arange(half, device=t.device, dtype=torch.float32)
            / (half - 1)
        )

        angles = t.float()[:, None] * frequencies[None, :]

        return torch.cat([angles.sin(), angles.cos()], dim=1)


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU()
        )

        self.time_projection = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU()
        )

    def forward(self, x, time_embedding):
        h = self.conv1(x)

        # Une information temporelle par canal.
        time_features = self.time_projection(time_embedding)
        h = h + time_features[:, :, None, None]

        return self.conv2(h)


class SmallUNet(nn.Module):
    def __init__(self, time_dim=128):
        super().__init__()

        self.time_embedding = nn.Sequential(
            TimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.encoder1 = ConvBlock(1, 32, time_dim)
        self.encoder2 = ConvBlock(32, 64, time_dim)
        self.middle = ConvBlock(64, 128, time_dim)

        self.pool = nn.AvgPool2d(2)

        self.decoder2 = ConvBlock(128 + 64, 64, time_dim)
        self.decoder1 = ConvBlock(64 + 32, 32, time_dim)

        self.output = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x, t):
        time_embedding = self.time_embedding(t)

        h1 = self.encoder1(x, time_embedding)               # 28 × 28
        h2 = self.encoder2(self.pool(h1), time_embedding)   # 14 × 14
        h = self.middle(self.pool(h2), time_embedding)      # 7 × 7

        h = F.interpolate(h, size=h2.shape[-2:], mode="nearest")
        h = self.decoder2(torch.cat([h, h2], dim=1), time_embedding)

        h = F.interpolate(h, size=h1.shape[-2:], mode="nearest")
        h = self.decoder1(torch.cat([h, h1], dim=1), time_embedding)

        return self.output(h)


diffusion_model = SmallUNet().to(device)
print(f"Paramètres : {sum(p.numel() for p in diffusion_model.parameters()):,}")

## 5. Apprendre à prédire le bruit

Pour chaque lot : tirer un temps aléatoire par image, bruiter les images avec `add_noise`, prédire le bruit,
et minimiser $\mathcal{L} = MSE(\varepsilon_\theta(x_t,t), \varepsilon)$.

La cible n'est ni le label ni l'image originale, mais **le bruit**. Tirer les temps au hasard permet d'apprendre
tous les niveaux de bruit sans parcourir les 200 étapes à chaque itération. `clip_grad_norm_` limite la norme
des gradients pour stabiliser l'entraînement.

In [ ]:
optimizer = torch.optim.Adam(diffusion_model.parameters(), lr=2e-4)


def train_diffusion(model, loader, optimizer, epochs=20):
    history = []

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        n_images = 0

        for x, _ in loader:
            x = x.to(device)
            batch_size = x.size(0)

            # Un niveau de bruit différent pour chaque image.
            t = torch.randint(0, T, (batch_size,), device=device)

            xt, true_noise = add_noise(x, t)
            predicted_noise = model(xt, t)

            loss = F.mse_loss(predicted_noise, true_noise)

            optimizer.zero_grad()
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * batch_size
            n_images += batch_size

        mean_loss = total_loss / n_images
        history.append(mean_loss)

        print(f"Epoch {epoch + 1}/{epochs} | MSE bruit = {mean_loss:.4f}")

    return history


history = train_diffusion(diffusion_model, train_loader, optimizer, epochs=20)
torch.save(diffusion_model.state_dict(), "diffusion_mnist.pt")

## 6. Générer des images par diffusion inverse

On part d'un bruit pur $x_T$, puis pour chaque étape de $T-1$ à 0 :
1. le réseau prédit le bruit présent ;
2. on en déduit une estimation de l'image propre $\hat x_0$, limitée à [−1, 1] ;
3. on calcule l'image de l'étape précédente (moyenne + un peu de bruit, sauf à la dernière étape).

In [ ]:
@torch.no_grad()
def generate_images(model, n_samples=20):
    model.eval()
    model_device = next(model.parameters()).device

    # Réutiliser exactement le calendrier de l'entraînement.
    b = betas.to(model_device)
    a = 1.0 - b
    abar = torch.cumprod(a, dim=0)

    abar_prev = torch.cat([torch.ones(1, device=model_device), abar[:-1]])

    # Variance du processus inverse.
    posterior_var = b * (1.0 - abar_prev) / (1.0 - abar)

    # Coefficients de la moyenne conditionnelle q(x_{t-1} | x_t, x_0).
    coef_x0 = b * torch.sqrt(abar_prev) / (1.0 - abar)
    coef_xt = torch.sqrt(a) * (1.0 - abar_prev) / (1.0 - abar)

    x = torch.randn(n_samples, 1, 28, 28, device=model_device)

    for step in reversed(range(len(b))):
        t = torch.full((n_samples,), step, device=model_device, dtype=torch.long)

        predicted_noise = model(x, t)

        # Estimation de l'image propre.
        x0_pred = (x - torch.sqrt(1.0 - abar[step]) * predicted_noise) / torch.sqrt(abar[step])

        # Les images propres d'entraînement sont dans [-1, 1].
        x0_pred = x0_pred.clamp(-1.0, 1.0)

        # Moyenne de l'étape précédente.
        mean = coef_x0[step] * x0_pred + coef_xt[step] * x

        if step > 0:
            x = mean + torch.sqrt(posterior_var[step]) * torch.randn_like(x)
        else:
            x = mean

        if not torch.isfinite(x).all():
            raise RuntimeError(f"Valeurs non finies pendant la génération, étape {step}")

    return ((x + 1.0) / 2.0).clamp(0, 1).cpu()

## 7. Visualiser les nouvelles images

In [ ]:
samples = generate_images(diffusion_model, n_samples=20)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i, 0], cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

plt.suptitle("Chiffres MNIST générés par diffusion")
plt.tight_layout()
plt.show()

Le modèle n'est pas conditionné par une classe : on ne choisit pas le chiffre généré, et les images changent
à chaque exécution.

## 8. Observer l'évolution de la perte

In [ ]:
plt.plot(range(1, len(history) + 1), history)
plt.xlabel("Époque")
plt.ylabel("MSE de prédiction du bruit")
plt.title("Entraînement du modèle de diffusion")
plt.grid(alpha=0.3)
plt.show()

La perte baisse rapidement puis se stabilise : le réseau prédit de mieux en mieux le bruit. C'est une perte
d'entraînement : pour juger le modèle, il faut aussi regarder la qualité et la diversité des images générées.

## Bilan des cinq parties

| Modèle | Ce qu'il apprend | Génère de nouvelles images ? |
|---|---|---|
| Classifieur | $P(y \mid x)$ | non |
| Autoencodeur | reconstruire $x$ via un code $z$ | mal : espace latent irrégulier |
| VAE | une distribution latente proche de $\mathcal N(0,I)$ | oui, images un peu floues |
| GAN | tromper un discriminateur | oui, images nettes, entraînement instable |
| Diffusion | prédire le bruit à chaque niveau | oui, très bonne qualité, génération lente (200 étapes) |